In [ ]:
import glob
from pathlib import Path
import numpy as np
import pandas as pd

# 1. Definindo os caminhos com base na sua estrutura
pasta_bruta = Path("C:\\Users\\Usuário\\Desktop\\Dengue_Alerta\\data\\dados\\bruto\\inmet")
pasta_processada = Path("C:\Users\Usuário\Desktop\Dengue_Alerta\data\processed")
pasta_processada.mkdir(parents=True, exist_ok=True)

# 2. Buscando todos os arquivos CSV dentro das pastas de 2020 a 2025
arquivos_inmet = list(pasta_bruta.glob("**/*.CSV")) + list(
    pasta_bruta.glob("**/*.csv")
)

print(f"Total de arquivos INMET encontrados: {len(arquivos_inmet)}")


# 3. Função para carregar e limpar um arquivo individual do INMET
def limpar_arquivo_inmet(caminho_arquivo):
  try:
    # O INMET costuma usar encoding 'latin1' ou 'iso-8859-1' e separador ';'
    # skiprows=8 pula o cabeçalho descritivo padrão do INMET (confirme abrindo um arquivo se necessário)
    df = pd.read_csv(
        caminho_arquivo,
        sep=";",
        encoding="latin1",
        skiprows=8,
        decimal=",",
        na_values=["-9999", ""],
    )

    # Padronizar nomes das colunas (remover espaços, colocar em maiúsculo)
    df.columns = (
        df.columns.str.strip().str.upper().str.replace(" ", "_").str.normalize("NFKD").str.encode("ascii", errors="ignore").str.decode("utf-8")
    )

    # Criar coluna de data unindo DATA (UTC) e HORA (UTC) se existirem
    if "DATA_(UTC)" in df.columns and "HORA_(UTC)" in df.columns:
      # A hora no INMET costuma vir como '0000 UTC', '1200 UTC', etc.
      df["HORA"] = df["HORA_(UTC)"].str.replace(" UTC", "").str[:2]
      df["DATA_HORA"] = pd.to_datetime(
          df["DATA_(UTC)"] + " " + df["HORA"] + ":00:00", errors="coerce"
      )
      df.drop(columns=["DATA_(UTC)", "HORA_(UTC)", "HORA"], inplace=True)

    return df
  except Exception as e:
    print(f"Erro ao processar {caminho_arquivo.name}: {e}")
    return None


# 4. Processando e concatenando todos os anos
lista_dfs = []
for arquivo in arquivos_inmet:
  df_temp = limpar_arquivo_inmet(arquivo)
  if df_temp is not None and not df_temp.empty:
    lista_dfs.append(df_temp)

if lista_dfs:
  df_inmet_bruto = pd.concat(lista_dfs, ignore_index=True)
  print(f"Dataset consolidado bruto: {df_inmet_bruto.shape}")
else:
  print("Nenhum dado foi carregado. Verifique o caminho dos arquivos.")

Total de arquivos INMET encontrados: 0
Nenhum dado foi carregado. Verifique o caminho dos arquivos.
